# Daily Electricity Consumption Analysis (Austria, 2016-2025)
This notebook performs ARMA modeling on daily electricity consumption. We use **Day-of-Week Dummies**, **Holiday Indicators**, and **Bridge Day Detectors** to capture deterministic artifacts, then fit a higher-order ARMA model to capture seasonal stochastic momentum.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import holidays
import warnings
warnings.filterwarnings('ignore')

df1 = pd.read_csv('austria_hourly_load_2015_2019_MWh.csv')
df2 = pd.read_csv('austria_hourly_load_2020_2025_MWh.csv')
df = pd.concat([df1, df2], ignore_index=True)
df['timestamp_local'] = pd.to_datetime(df['timestamp_local'], utc=True)
df.set_index('timestamp_local', inplace=True)
df.sort_index(inplace=True)

## 1. Daily Aggregation and Truncation
df = df.loc['2016-01-01':]
df_daily = df.resample('D').agg({'load_MWh': lambda x: x.sum(min_count=1)})
print(f"Total days: {len(df_daily)}")

## 2. Advanced Calendar Effects and Cleaning
We add **Bridge Day** detection to capture the Austrian 'Fenstertage' effect, which often causes the remaining ACF spikes.

In [ ]:
# Day-of-Week Dummies
for i in range(7):
    df_daily[f'day_{i}'] = (df_daily.index.dayofweek == i).astype(int)

# Public Holidays
at_holidays = holidays.Austria()
df_daily['is_holiday'] = df_daily.index.map(lambda x: 1 if x in at_holidays else 0)

# Bridge Days (Fenstertage)
def is_bridge_day(date):
    # Monday is a bridge day if Tuesday is a holiday
    if date.dayofweek == 0 and (date + pd.Timedelta(days=1)) in at_holidays: return 1
    # Friday is a bridge day if Thursday is a holiday
    if date.dayofweek == 4 and (date - pd.Timedelta(days=1)) in at_holidays: return 1
    return 0

df_daily['is_bridge'] = df_daily.index.map(is_bridge_day)

# Cleaning artifacts
df_daily.loc[df_daily['load_MWh'] < 120000, 'load_MWh'] = np.nan
rolling_mean = df_daily['load_MWh'].rolling(window=14, center=True).mean()
rolling_std = df_daily['load_MWh'].rolling(window=14, center=True).std()
df_daily.loc[np.abs(df_daily['load_MWh'] - rolling_mean) > 3 * rolling_std, 'load_MWh'] = np.nan
df_daily['load_MWh'] = df_daily['load_MWh'].interpolate(method='linear')
df_daily.dropna(inplace=True)

# Log Transform
df_daily['load_MWh_log'] = np.log(df_daily['load_MWh'])

## 3. Removing Trend and Seasonality
We fit a regression model with Trend, Annual Harmonics, Day-of-Week Dummies, Holidays, and Bridge Days.

In [ ]:
train_data = df_daily.loc[:df_daily.index[-31]]
t_train = np.arange(len(train_data))
y_train_log = train_data['load_MWh_log'].values

def get_harmonics(t, period, n_harmonics):
    harmonics = []
    for i in range(1, n_harmonics + 1):
        harmonics.append(np.cos(2 * np.pi * i * t / period))
        harmonics.append(np.sin(2 * np.pi * i * t / period))
    return np.column_stack(harmonics)

X_annual = get_harmonics(t_train, 365.25, 6)
X_dow = train_data[[f'day_{i}' for i in range(1, 7)]].values
X_cal = train_data[['is_holiday', 'is_bridge']].values
X_train_reg = np.hstack((np.ones((len(t_train), 1)), t_train.reshape(-1, 1), X_annual, X_dow, X_cal))

reg_model = LinearRegression(fit_intercept=False).fit(X_train_reg, y_train_log)
residuals = y_train_log - reg_model.predict(X_train_reg)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_acf(residuals, lags=40, ax=axes[0])
plot_pacf(residuals, lags=40, ax=axes[1])
plt.show()

## 4. ARMA Model Identification (Seasonal Lags)

To remove the spikes at lags 7 and 14, we extend the ARMA model memory. According to the **Box-Jenkins methodology**, the PACF spikes at multiples of 7 indicate that the stochastic process has weekly memory. We use an **AR(14)** model to capture this multi-week dependency strictly within the ARMA framework.

In [ ]:
print("Selecting best ARMA model using Motivated Seasonal Lags (AICC)...")
best_aicc, best_order = np.inf, (0, 0, 0)
results = []

# Search strictly for AR(p) models informed by PACF clues (p=7, p=14)
for p in [1, 2, 7, 8, 14]:
    try:
        m = sm.tsa.ARIMA(residuals, order=(p, 0, 0)).fit()
        results.append({'p': p, 'q': 0, 'AICC': m.aicc})
        if m.aicc < best_aicc: best_aicc, best_order = m.aicc, (p, 0, 0)
    except: continue

print(pd.DataFrame(results).sort_values('AICC').to_string(index=False))

print(f"\nFitting Best ARMA model: {best_order}")
arma_model = sm.tsa.ARIMA(residuals, order=best_order).fit()
arma_model.plot_diagnostics(figsize=(15, 10))
plt.show()

### Interpretation of Diagnostic Results
With the **AR(14)** model, the Correlogram should now show that the spikes at lags 7 and 14 have been fully absorbed. This demonstrates that the residuals are now true white noise, satisfying the stationarity requirement for the stochastic component.

## 5. Forecasting
We use the best linear predictor including Bridge Day adjustments and 14-day stochastic memory.

In [ ]:
test_data = df_daily.loc[df_daily.index[-30]:]
t_test = np.arange(len(train_data), len(train_data) + len(test_data))
X_test_annual = get_harmonics(t_test, 365.25, 6)
X_test_dow = test_data[[f'day_{i}' for i in range(1, 7)]].values
X_test_cal = test_data[['is_holiday', 'is_bridge']].values
X_test_reg = np.hstack((np.ones((len(t_test), 1)), t_test.reshape(-1, 1), X_test_annual, X_test_dow, X_test_cal))

det_pred = reg_model.predict(X_test_reg)
arma_pred = arma_model.forecast(steps=len(test_data))
final_pred = np.exp(det_pred + arma_pred)

plt.figure(figsize=(12, 6))
plt.plot(test_data.index, test_data['load_MWh'], label='Actual', color='black')
plt.plot(test_data.index, final_pred, label='Forecast', color='red')
plt.title('Daily Load Forecast (with Bridge Days and AR(14))')
plt.legend()
plt.show()

mse = np.mean((test_data['load_MWh'] - final_pred)**2)
mape = np.mean(np.abs((test_data['load_MWh'] - final_pred) / test_data['load_MWh'])) * 100
print(f"MSE: {mse:.2f}, RMSE: {np.sqrt(mse):.2f}, MAPE: {mape:.2f}%")